# 🎬 AI Video Summarizer
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/video-summarizer/training/notebook.ipynb)

Turn a video or audio file into a **timestamped transcript, a summary and 5 key points**. The audio
is decoded with PyAV (16 kHz mono, first 180 s), transcribed by **Whisper** (transformers ASR pipeline)
and summarized by a small instruction-tuned LLM (**Qwen2.5-Instruct**) that answers in JSON.
Nothing is trained: this notebook runs the exact pipeline of the
[Space](https://huggingface.co/spaces/shalev396/video-summarizer) (`../space/pipeline.py`) on sample media,
evaluates it (WER, speed, summary length) and exports the Space examples + `results.json`.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training (none: run the pipeline) ·
6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / space code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "video-summarizer"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../space/pipeline.py").exists():   # space/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/space"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
import json

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../space on sys.path
from src.config import Config
import pipeline as P                                                 # ../space/pipeline.py

print(utils.lib_versions("torch", "transformers", "av", "huggingface_hub", "numpy"))

## 2. Config
Every knob lives in `src/config.py`. Two model pairs (variants) are evaluated:
`cpu` = whisper-tiny + Qwen2.5-0.5B-Instruct (what CPU hardware runs) and `gpu` = whisper-small +
Qwen2.5-1.5B-Instruct (what the ZeroGPU Space runs). `SMOKE_TEST=1` evaluates only the small pair and
writes everything to `outputs/smoke/`.

In [ ]:
import os

os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
utils.set_seeds(42, "torch")

cfg = Config()
print("smoke:", cfg.smoke, "| outputs:", cfg.outputs_dir.relative_to(utils.PROJECT_DIR),
      "| examples ->", cfg.examples_dir.relative_to(utils.PROJECT_DIR))
cfg

## 3. Data
Three inputs, cached in `training/data/`: the 11 s JFK clip (`jfk.wav`, downloaded from whisper.cpp,
with openai/whisper's `jfk.flac` as fallback), the same audio inside an H.264/AAC video (`sample.mp4`,
built with PyAV when missing) and a short lecture transcript for the paste-a-transcript path.
`P.load_audio` is the decoder the Space uses: any audio/video file -> 16 kHz mono float32, cut at 180 s.

In [ ]:
samples = data_setup.ensure_samples(cfg)
pd.DataFrame(data_setup.describe(samples, cfg.max_seconds))

In [ ]:
audio, duration = P.load_audio(samples["video"], cfg.max_seconds)
t = [i / P.SAMPLE_RATE for i in range(audio.size)]
plt.figure(figsize=(10, 2.5))
plt.plot(t[::20], audio[::20], lw=0.5)
plt.title(f"{samples['video'].name}: decoded audio ({duration:.1f} s, {P.SAMPLE_RATE} Hz mono)")
plt.xlabel("seconds")
plt.show()
print("reference transcript:", cfg.reference)

## 4. Load model
`model_builder.build_pipeline` builds `pipeline.Pipeline` from `../space/pipeline.py` with the Hub
checkpoints of a variant (downloaded once into the Hugging Face cache). fp16 on CUDA, fp32 elsewhere.

In [ ]:
first = next(iter(cfg.variants))
pipe = model_builder.build_pipeline(cfg, device, first)
pd.DataFrame(model_builder.describe(pipe))

## 5. Training
There is nothing to train: every component is a pretrained checkpoint. This section runs each variant
on the samples **stage by stage** (decode -> transcribe -> LLM JSON reply -> parse) and records the
timings. The full run (both variants) takes a few minutes on a CPU; the first run also downloads the
checkpoints (~1 GB for whisper-small + 3 GB for Qwen2.5-1.5B).

In [ ]:
results, params = {}, {}
for variant in cfg.variants:
    if pipe is None or (pipe.asr_id, pipe.llm_id) != cfg.variants[variant]:
        pipe = model_builder.build_pipeline(cfg, device, variant)
    print(f"--- {variant}: {pipe.asr_id} + {pipe.llm_id} on {device}")
    params[variant] = model_builder.describe(pipe)
    results[variant] = engine.run_samples(pipe, samples, cfg.reference)
    pipe = None                                           # free memory before the next variant

In [ ]:
out = results[first][0]["output"]
print("transcript:", out["transcript"])
print("segments:", out["segments"])
print("summary:", out["summary"])
for point in out["key_points"]:
    print(" -", point)

## 6. Evaluation
- **WER** of the transcript vs the known words of the JFK clip (lower-cased, punctuation removed),
  averaged over the WAV and the MP4.
- **RTF** (real-time factor) = ASR seconds / audio seconds (< 1 = faster than real time).
- **LLM seconds**, **summary words**, **key points** (target 5) and **JSON rate** = share of LLM replies
  that were valid JSON before the lenient fallback parser.
Timings are for the device shown; the `gpu` variant timed on a CPU is much slower than on ZeroGPU.

In [ ]:
table = engine.evaluate(results)
pd.DataFrame(table).T.round(3)

In [ ]:
per_run = pd.DataFrame([{"variant": v, **{k: r[k] for k in r if k != "output"}}
                        for v, rows in results.items() for r in rows])
plot_path = export.plot_timings(results, cfg.outputs_dir / "assets" / "timings.png")
display(ShowImage(filename=str(plot_path)))
per_run.round(3)

In [ ]:
for variant, rows in results.items():
    lecture = rows[-1]["output"]
    print(f"--- {variant} on {rows[-1]['sample']}\n{lecture['summary']}")
    for point in lecture["key_points"]:
        print(" -", point)

## 7. Inference
Exactly what the Space runs: `P.load(device).predict(media=..., transcript=...)` returns the `/predict`
JSON `{language, transcript, segments, summary, key_points, models}`. On a CUDA machine this picks the
`gpu` pair, elsewhere the `cpu` pair.

In [ ]:
examples_dir = utils.SPACE_DIR / "examples" if (utils.SPACE_DIR / "examples").is_dir() else cfg.data_dir
predictor = P.load(device)
print("models:", predictor.models)
result = predictor.predict(media=str(examples_dir / cfg.video_sample))
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
result = predictor.predict(transcript=(examples_dir / cfg.transcript_sample).read_text(encoding="utf-8"))
print(json.dumps({k: result[k] for k in ("summary", "key_points", "models")}, indent=2, ensure_ascii=False))

## 8. Export
No weights to save (the model repo `../model` only holds a README listing the pretrained models).
Export writes the Space examples (`../space/examples/`; smoke: `outputs/smoke/examples/`) and
`results.json` + plots into `training/outputs/` (smoke: `outputs/smoke/`).

In [ ]:
written = export.write_examples(samples, cfg.examples_dir)
payload = export.build_results(cfg, results, table, params, device)
path = export.save_results(payload, cfg.outputs_dir)
for p in written + [path, plot_path]:
    print(f"{p.relative_to(utils.PROJECT_DIR).as_posix():45s} {p.stat().st_size / 1024:8.1f} KB")